### MiddleWare

- Tracking agent Behaviour with logging,analytics and debugging
- Transforming prompts, tool selection & output formatting

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()


os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")



### Summarization Middleware

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model

model=init_chat_model("groq:openai/gpt-oss-120b")

### Messagebased summarization

### trigger based length of the message
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
### run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [4]:
# test data

questions=[
    "What is 2+2?",
    "What is 10*5?",
    "What is 21%2?",
    "What is 1123/5?",
    "What is 2980+234?",
    "What is 10*50?",
    "What is 223445/2903?",
    "What is 1045-50?",
    "What is 105-50?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='a373ff06-e666-4d3e-8347-12f5d61fd54e'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question. Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 78, 'total_tokens': 111, 'completion_time': 0.073014563, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.003191178, 'prompt_tokens_details': None, 'queue_time': 0.045387111, 'total_time': 0.076205741}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_76082d2bd0', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12226-287b-7ef1-9c2d-5ad3ff5aa10c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 33, 'total_tokens': 111, 'output_token_details': {'reasoning': 14}})]}
Messages: 2
M

### trigger based on token size

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model

model=init_chat_model("groq:openai/gpt-oss-120b")

@tool
def search_hotels(city:str)->str:
    """Search Hotels - return long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star ,$350/night, spa, pool, gym
    2. City Inn - 4 star ,$180/night, business center
    3. Budget stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)


config={"configurable":{"thread_id":"test-1"}}


def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars

In [6]:
cities=["Chennai","Mumbai","Bangalore","Hyderbad","Delhi","Hubli","Ranebennur"]

for city in cities:
    response=agent.invoke({
        "messages":[HumanMessage(content=f"Find hotels in {city}")]
    },config)
    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens}tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Chennai: ~1179tokens, 4 messages
[HumanMessage(content='Find hotels in Chennai', additional_kwargs={}, response_metadata={}, id='1a0308e0-8ff5-4594-aea7-73064fae7b0b'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants hotels in Chennai. Use function search_hotels.', 'tool_calls': [{'id': 'fc_79693529-0198-43d3-b3be-45bbbdb7e4cd', 'function': {'arguments': '{"city":"Chennai"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 129, 'total_tokens': 172, 'completion_time': 0.090920796, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.005133775, 'prompt_tokens_details': None, 'queue_time': 0.392420387, 'total_time': 0.096054571}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8655ddce88', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12226-4ab3-7c70-93a4-457fb86b1592-0', tool

### Fraction

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model

model=init_chat_model("groq:openai/gpt-oss-120b")

@tool
def search_hotels(city:str)->str:
    """Search Hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget stay $75"


agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction",0.005),
            keep=("fraction",0.002),
        )
    ]
)


config={"configurable":{"thread_id":"test-1"}}


def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages)

cities=["Chennai","Mumbai","Bangalore","Hyderbad","Delhi","Hubli","Ranebennur"]

for city in cities:
    response=agent.invoke({
        "messages":[HumanMessage(content=f"hotels in {city}")]
    },config)
    tokens=count_tokens(response["messages"])
    fraction=tokens/128000
    print(f"{city}: ~{tokens}tokens ({fraction:.4%}), {len(response['messages'])} messages")
    print(f"{(response['messages'])}")
    

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m4bzbwz5ey1bgzh04n5r77k1` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199859, Requested 800. Please try again in 4m44.688s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### HumanInTheLoopMiddleware

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

from langchain.chat_models import init_chat_model

model=init_chat_model("groq:openai/gpt-oss-120b")

def read_email_tool(email_id: str)-> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recepient: str,subject:str, body:str)->str:
    """Mock function to send an email."""
    return f"Email sent to {recepient} with subject '{subject}'"

In [ ]:
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [ ]:
config={"configurable":{"thread_id":"test_approve"}}

result=agent.invoke({
    "messages":[HumanMessage(content="Send email to surveabhishek68@gmail.com with subject 'Hello' and body how are you?")]
},config)

In [ ]:
result

In [ ]:
#approval
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving...")

    result=agent.invoke(Command(resume={
        "decisions":[
            {"type":"approve"}
        ]
    }),config)

    print(f"Result: {result['messages'][-1].content}")

In [ ]:
result

### Reject

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

from langchain.chat_models import init_chat_model

model=init_chat_model("groq:openai/gpt-oss-120b")

def read_email_tool(email_id: str)-> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recepient: str,subject:str, body:str)->str:
    """Mock function to send an email."""
    return f"Email sent to {recepient} with subject '{subject}'"

agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)


In [ ]:
config={"configurable":{"thread_id":"test_reject"}}

result=agent.invoke({
    "messages":[HumanMessage(content="Send email to surveabhishek68@gmail.com with subject 'Hello' and body how are you?")]
},config)

In [ ]:
result

In [ ]:
#approval
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving...")

    result=agent.invoke(Command(resume={
        "decisions":[
            {"type":"reject"}
        ]
    }),config)

    print(f"Result: {result['messages'][-1].content}")

In [ ]:
result

### Editing

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

from langchain.chat_models import init_chat_model

model=init_chat_model("groq:openai/gpt-oss-120b")

def read_email_tool(email_id: str)-> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recepient: str,subject:str, body:str)->str:
    """Mock function to send an email."""
    return f"Email sent to {recepient} with subject '{subject}'"

agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)


In [ ]:
config={"configurable":{"thread_id":"test_edit"}}

result=agent.invoke({
    "messages":[HumanMessage(content="Send email to surveabhishek68@gmail.com with subject 'Hello' and body how are you?")]
},config)

In [ ]:
result

In [ ]:
#approval
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving...")

    result=agent.invoke(Command(resume={
        "decisions":[
            {
                "type":"edit",
                "edited_action":{
                    "name":"send_email_tool",
                    "args":{
                        "recepient":"corrected@gmail.com",
                        "subject":"Corrected Subject",
                        "body":"This was edited by human before editing"
                    }
                }
            }
        ]
    }),config)

    print(f"Result: {result['messages'][-1].content}")

In [ ]:
result